In [1]:
import pandas as pd 
import re 
import html 
from collections import Counter 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:

df = pd.read_csv('apple_tweets.csv') 
apple_responses = df[df['author_id'] == 'AppleSupport'].copy() 
print(f'Total de respuestas: {len(apple_responses)}')


Total de respuestas: 106860


In [5]:
stopwords_comunes = { 'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'you', 'your', 'yours', 
                     'he', 'she', 'it', 'they', 'them', 'what', 'which', 'who', 'this', 'that', 
                     'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 
                     'have', 'has', 'had', 'do', 'does', 'did', 'a', 'an', 'the', 'and', 'but', 
                     'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 
                     'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 
                     'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 
                     'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 
                     'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 
                     'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 
                     'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now', 'gt', 
                     'lt', 'amp' }

def limpiar_texto(texto): 
    if not isinstance(texto, str): 
        return [] 
    texto = texto.lower() 
    texto = re.sub(r'&amp;(?:gt|lt|amp|quot|apos);', ' ', texto) 
    texto = re.sub(r'https?://[^ \\t\\n\\r]+|www.[^ \\t\\n\\r]+', '', texto) 
    texto = re.sub(r'@[a-zA-Z0-9\_]+', '', texto) 
    texto = re.sub(r'[^a-z \\t\\n\\r]', ' ', texto) 
    palabras = texto.split() 
    return [p for p in palabras if p not in stopwords_comunes and len(p) > 2]

apple_responses['palabras'] = apple_responses['text'].apply(limpiar_texto) 
apple_responses['texto_limpio'] = apple_responses['palabras'].apply(lambda x: ' '.join(x))

todas_las_palabras = [palabra for sublista in apple_responses['palabras'] for palabra in sublista] 
diccionario_frecuencias = Counter(todas_las_palabras)

print('--- TOP 20 PALABRAS MÁS FRECUENTES ---') 
for palabra, frecuencia in diccionario_frecuencias.most_common(20): print(f'{palabra:<15}: {frecuencia} apariciones')

--- TOP 20 PALABRAS MÁS FRECUENTES ---
https          : 87473 apariciones
gdrqu          : 55667 apariciones
ypt            : 53988 apariciones
help           : 40232 apariciones
let            : 33222 apariciones
look           : 21684 apariciones
ios            : 21246 apariciones
know           : 19489 apariciones
issue          : 16782 apariciones
version        : 16654 apariciones
iphone         : 15907 apariciones
please         : 15792 apariciones
like           : 14983 apariciones
get            : 14800 apariciones
send           : 13479 apariciones
update         : 12848 apariciones
work           : 12084 apariciones
device         : 11844 apariciones
using          : 11517 apariciones
thanks         : 11315 apariciones


In [4]:
vectorizer = TfidfVectorizer() 
tfidf_matrix = vectorizer.fit_transform(apple_responses['texto_limpio'])

N = tfidf_matrix.shape[0]
col_sums = np.asarray(tfidf_matrix.sum(axis=0)).ravel()
suma_total = np.dot(col_sums, col_sums)
suma_sin_diagonal = suma_total - N
similitud_promedio = suma_sin_diagonal / (N * (N - 1))
print(f"Similitud promedio: {similitud_promedio:.4f}")

#matriz_similitud = cosine_similarity(tfidf_matrix) 
#np.fill_diagonal(matriz_similitud, np.nan) 
#similitud_promedio = np.nanmean(matriz_similitud)

#print(f'Similitud coseno promedio: {similitud_promedio:.4f}')

Similitud promedio: 0.0575


In [7]:
a= apple_responses.sample(n=100)
a.to_csv('muestra_apple_responses.csv', index=False)